<a href="https://colab.research.google.com/github/rodrigoiyg27/TopicosEspeciales2026/blob/main/Clase6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

# Descargar la base SQLite Chinook desde GitHub
!wget -q https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite -O chinook.sqlite

con = sqlite3.connect("chinook.sqlite")

# Leer cada tabla con SQL.
clientes  = pd.read_sql("SELECT * FROM Customer", con)
#parse_dates convierte InvoiceDate de texto a fecha
facturas  = pd.read_sql("SELECT * FROM Invoice", con, parse_dates=["InvoiceDate"])
lineas    = pd.read_sql("SELECT * FROM InvoiceLine", con)
canciones = pd.read_sql("SELECT * FROM Track", con)

print("Facturas", facturas.shape)
print("Clientes", clientes.shape)
print("Lineas", lineas.shape)
print("Canciones", canciones.shape)

Facturas (412, 9)
Clientes (59, 13)
Lineas (2240, 5)
Canciones (3503, 9)


Verificar claves

In [ ]:
# Paso 1: verificar que cada clave primaria sea única
for nombre, df, clave in [("clientes", clientes, "CustomerId"),
                          ("facturas", facturas, "InvoiceId"),
                          ("lineas", lineas, "InvoiceLineId"),
                          ("canciones", canciones, "TrackId")]:
    print(f"{nombre}: clave {clave} única -> {df[clave].is_unique}")

# Paso 2: anti-join. ¿Qué canciones del catálogo nunca se vendieron?
# drop_duplicates
vendidas = lineas[["TrackId"]].drop_duplicates()

cobertura = canciones[["TrackId", "Name"]].merge(
    vendidas, on="TrackId",
    how="left",           # conservar TODAS las canciones del catálogo
    indicator="origen"    # both = se vendió, left_only = nunca se vendió
)
nunca_vendidas = cobertura.query("origen == 'left_only'")
print("Canciones que nunca se vendieron")
print(f"{len(nunca_vendidas)} de {len(canciones)} ({len(nunca_vendidas)/len(canciones):.1%})")
print(nunca_vendidas)

clientes: clave CustomerId única -> True
facturas: clave InvoiceId única -> True
lineas: clave InvoiceLineId única -> True
canciones: clave TrackId única -> True
Canciones que nunca se vendieron
1519 de 3503 (43.4%)
      TrackId                                               Name     origen
6           7                                    Let's Get It Up  left_only
10         11                                             C.O.D.  left_only
16         17                                  Let There Be Rock  left_only
17         18                                     Bad Boy Boogie  left_only
21         22                                  Whole Lotta Rosie  left_only
...       ...                                                ...        ...
3496     3497                                    Erlkonig, D.328  left_only
3497     3498  Concerto for Violin, Strings and Continuo in G...  left_only
3500     3501               L'orfeo, Act 3, Sinfonia (Orchestra)  left_only
3501     3502  Quintet f

Conciliación por totales

In [ ]:
# Total calculado desde el detalle: una fila por factura
total_lineas = (
    lineas.assign(Importe=lineas["UnitPrice"] * lineas["Quantity"])
          .groupby("InvoiceId", as_index=False)["Importe"].sum()
          .rename(columns={"Importe": "TotalLineas"})
)

cuadre = facturas[["InvoiceId", "Total"]].merge(
    total_lineas, on="InvoiceId",
    how="outer",               # detectar facturas sin líneas o líneas sin factura
    indicator=True,
    validate="one_to_one"      # si la clave se repite, falla en vez de inflar totales
)
cuadre["Diferencia"] = (cuadre["Total"] - cuadre["TotalLineas"]).round(2)

print(cuadre["_merge"].value_counts())
print("Facturas que no cuadran:", (cuadre["Diferencia"].abs() > 0.005).sum())
print("Gran total:", round(cuadre["Total"].sum(), 2), "vs", round(cuadre["TotalLineas"].sum(), 2))

_merge
both          412
left_only       0
right_only      0
Name: count, dtype: int64
Facturas que no cuadran: 0
Gran total: 2328.6 vs 2328.6


Comparación de valores entre tablas

In [ ]:
# ¿El precio cobrado en cada línea coincide con el precio de catálogo de la canción?
precios = lineas.merge(
    canciones[["TrackId", "UnitPrice"]], on="TrackId", how="left",
    suffixes=("_venta", "_catalogo"),
    validate="many_to_one"   # muchas líneas por canción, una canción por TrackId
)
# np.isclose: compara decimales con tolerancia
precios["coincide"] = np.isclose(precios["UnitPrice_venta"],
                                 precios["UnitPrice_catalogo"], atol=0.005)
print(precios["coincide"].value_counts())

# ¿El país de facturación coincide con el país registrado del cliente?
direcciones = facturas.merge(clientes[["CustomerId", "Country"]],
                             on="CustomerId", validate="many_to_one")
print("País distinto:", (direcciones["BillingCountry"] != direcciones["Country"]).sum())

coincide
True    2240
Name: count, dtype: int64
País distinto: 0


Verificación de exportación

Errores introducidos (con la semilla 42): faltan las facturas 36, 313 y 409; se duplican 179 y 266; la 177 tiene ruido de redondeo; 39, 83, 286 y 351 tienen montos alterados; y se agrega la 9999.

> Agregar bloque entrecomillado



In [ ]:
# --- Construcción de la copia
rng = np.random.default_rng(42)   # siempre los mismos IDs
contable = facturas[["InvoiceId", "CustomerId", "InvoiceDate", "Total"]].copy()

ids = rng.choice(contable["InvoiceId"].to_numpy(), size=10, replace=False)
faltantes, duplicadas, redondeo, alteradas = ids[:3], ids[3:5], ids[5:6], ids[6:10]

contable = contable[~contable["InvoiceId"].isin(faltantes)]                    # 3 no llegaron
contable = pd.concat([contable, contable[contable["InvoiceId"].isin(duplicadas)]],
                     ignore_index=True)                                         # 2 se cargaron dos veces
contable.loc[contable["InvoiceId"].isin(redondeo), "Total"] += 0.001            # 1 con ruido de redondeo
contable.loc[contable["InvoiceId"].isin(alteradas), "Total"] += np.array([1.00, -0.99, 5.94, 0.10])  # 4 mal digitadas
extra = pd.DataFrame({"InvoiceId": [9999], "CustomerId": [1],
                      "InvoiceDate": [pd.Timestamp("2025-12-23")], "Total": [3.96]})
contable = pd.concat([contable, extra], ignore_index=True)                     # 1 que no existe en ventas

# El otro sistema usa nombres en español
contable = contable.rename(columns={"InvoiceId": "factura_id", "CustomerId": "cliente_id",
                                    "InvoiceDate": "fecha", "Total": "monto"})

Verificar la clave

In [ ]:
# Mapeo
mapa = {"factura_id": "InvoiceId", "cliente_id": "CustomerId",
        "fecha": "InvoiceDate", "monto": "Total"}
contable = contable.rename(columns=mapa)

# ¿La clave es única en la fuente contable?
print(contable[contable.duplicated("InvoiceId", keep=False)].sort_values("InvoiceId"))

# validate convierte el supuesto "clave única" en una verificación automática
try:
    facturas.merge(contable, on="InvoiceId", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)

     InvoiceId  CustomerId InvoiceDate  Total
177        179          20  2023-02-20   8.91
409        179          20  2023-02-20   8.91
264        266          28  2024-03-24   1.98
410        266          28  2024-03-24   1.98
MergeError: Merge keys are not unique in right dataset; not a one-to-one merge


In [ ]:
contable_limpio = contable.drop_duplicates()   # solo filas idénticas en TODAS las columnas

Comparar valores

In [ ]:
comp = facturas[["InvoiceId", "CustomerId", "InvoiceDate", "Total"]].merge(
    contable_limpio, on="InvoiceId", how="outer",
    suffixes=("_ventas", "_contable"), indicator="origen", validate="one_to_one"
)

# Comparación con y sin tolerancia
ambos = comp.query("origen == 'both'").copy()
exacto   = (ambos["Total_ventas"] == ambos["Total_contable"])
tolerado = np.isclose(ambos["Total_ventas"], ambos["Total_contable"], atol=0.005)
print("Distintas con == :", (~exacto).sum())
print("Distintas con tolerancia:", (~tolerado).sum())

# Clasificación final: np.select evalúa en orden y gana la primera condición verdadera
comp["estado"] = np.select(
    [comp["origen"] == "left_only",
     comp["origen"] == "right_only",
     np.isclose(comp["Total_ventas"], comp["Total_contable"], atol=0.005)],
    ["Falta en contabilidad", "No existe en ventas", "Conciliada"],
    default="Monto distinto"
)
print(comp["estado"].value_counts())

Distintas con == : 5
Distintas con tolerancia: 4
estado
Conciliada               405
Monto distinto             4
Falta en contabilidad      3
No existe en ventas        1
Name: count, dtype: int64


Conciliación de totales mensuales

In [ ]:
def por_mes(df):
    """Conteo de facturas y monto total por mes."""
    return (df.assign(Mes=df["InvoiceDate"].dt.to_period("M"))
              .groupby("Mes").agg(facturas=("Total", "size"), monto=("Total", "sum")))

control = (por_mes(facturas)
           .join(por_mes(contable_limpio), lsuffix="_ventas", rsuffix="_contable", how="outer")
           .fillna(0))
control["dif_facturas"] = control["facturas_ventas"] - control["facturas_contable"]
control["dif_monto"] = (control["monto_ventas"] - control["monto_contable"]).round(2)

# De 60 meses, solo se revisan en detalle los que no cuadran
print(control.query("dif_facturas != 0 or abs(dif_monto) > 0.01"))

         facturas_ventas  monto_ventas  facturas_contable  monto_contable  \
Mes                                                                         
2021-06                7         37.62                  6           36.64   
2021-12                7         37.62                  7           36.63   
2024-06                7         37.62                  7           43.56   
2024-10                7         42.62                  6           25.76   
2025-03                7         37.62                  7           37.72   
2025-12                7         38.62                  7           36.64   

         dif_facturas  dif_monto  
Mes                               
2021-06             1       0.98  
2021-12             0       0.99  
2024-06             0      -5.94  
2024-10             1      16.86  
2025-03             0      -0.10  
2025-12             0       1.98  


Validación automática

In [ ]:
!pip install pandera
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

# Condiciones que debe cumplir cualquier tabla de facturas
esquema_facturas = DataFrameSchema({
    "InvoiceId":   Column(int, unique=True),                                   # sin duplicados
    "CustomerId":  Column(int, Check.isin(clientes["CustomerId"].tolist())),   # integridad referencial
    "InvoiceDate": Column(pa.DateTime, Check.le(pd.Timestamp("2025-12-31"))),  # no posterior al cierre
    "Total":       Column(float, Check.gt(0)),                                 # una factura no puede ser 0
})

esquema_facturas.validate(facturas[["InvoiceId", "CustomerId", "InvoiceDate", "Total"]])  # pasa

try:
    esquema_facturas.validate(contable, lazy=True)   # lazy=True: reporta TODOS los errores
except pa.errors.SchemaErrors as err:
    print(err.failure_cases[["column", "check", "index", "failure_case"]])

      column             check  index  failure_case
0  InvoiceId  field_uniqueness    177         179.0
1  InvoiceId  field_uniqueness    264         266.0
2  InvoiceId  field_uniqueness    409         179.0
3  InvoiceId  field_uniqueness    410         266.0
4      Total   greater_than(0)     81           0.0
